# 03 — Prepare
Add apples-to-apples comparison metrics and package the export.

**Why normalize?** Series differ in episode count (S1=6, S2–3=5, S4–5=8, S6–21=10), so more recent series have far more points on offer. Raw totals therefore can't be compared across series. We fix that by computing each contestant's **share of their series' points** and related normalized measures.

Metrics added (see `src/prepare.add_series_metrics`):
- `pct_of_series_points` — contestant points ÷ series total (**the % you asked for**)
- `share_vs_equal` — that share relative to an even 1/5 split (1.0 = average contestant; the cleanest cross-series yardstick)
- `points_vs_series_mean`, `series_rank`, `is_winner`, `pct_gap_to_winner`, plus series totals/means

In [ ]:
import sys, os
from pathlib import Path

PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

import pandas as pd
from src.ingest import load_config
from src.clean_quality import get_connection, load_to_duckdb, save_processed, run_sql
from src.prepare import add_series_metrics, package_dataset

cfg = load_config('config.yaml')
con = get_connection(cfg)
pd.set_option('display.float_format', lambda v: f'{v:,.4f}')
print(f'Project: {cfg["project_name"]}')

## Load the clean table
Falls back to the interim Parquet if the DuckDB table isn't present.

In [ ]:
tables = {r[0] for r in con.execute('SHOW TABLES').fetchall()}
if 'contestant_points_clean' in tables:
    df_clean = con.execute('SELECT * FROM contestant_points_clean').df()
else:
    df_clean = pd.read_parquet(Path(cfg['paths']['data_interim']) / 'contestant_points_clean.parquet')
print(df_clean.shape)
df_clean.head()

## Compute metrics

In [ ]:
df = add_series_metrics(df_clean)
df.head(10)

In [ ]:
# Validate: shares within each series must sum to 1.0 (100%).
# Columns are rounded to 4 dp for export, so allow a tiny rounding tolerance.
series_sum = df.groupby('series')['pct_of_series_points'].sum()
assert (series_sum - 1.0).abs().lt(1e-3).all(), 'series shares do not sum to 100%'

# Every series has exactly one rank-1 winner.
assert df.groupby('series')['is_winner'].sum().eq(1).all(), 'expected exactly one winner per series'

# share_vs_equal averages to 1.0 within each series by construction.
assert (df.groupby('series')['share_vs_equal'].mean() - 1.0).abs().lt(1e-3).all()

print('✓ Shares sum to 100% per series; one winner per series; share_vs_equal centers on 1.0.')

In [ ]:
# Who dominated their series the most / least, on a normalized basis?
cols = ['series', 'contestant', 'total_points', 'pct_of_series_points', 'share_vs_equal', 'series_rank']
print('Top 5 most dominant series performances (by share of series points):')
display(df.sort_values('pct_of_series_points', ascending=False)[cols].head(5))
print('\nBottom 5 (lowest share of series points):')
display(df.sort_values('pct_of_series_points')[cols].head(5))

## Persist processed table + save Parquet

In [ ]:
load_to_duckdb(df, 'contestant_metrics', con)
save_processed(df, cfg, 'contestant_metrics.parquet')

## Package the export (CSV + Excel + Parquet + codebook)

In [ ]:
codebook = {
    'series': 'Taskmaster UK series number (1-21).',
    'contestant': 'Contestant full name (corrected from first-name-only source against the official per-series cast).',
    'total_points': 'Raw total points the contestant earned across all tasks in their series, as adjudicated on-air. NOT comparable across series (see series_total_points).',
    'series_total_points': 'Sum of all points awarded to all five contestants in that series. Differs across series because episode/task counts differ (S1=6 episodes, S2-3=5, S4-5=8, S6-21=10).',
    'series_size': 'Number of contestants in the series (5 for the UK main run).',
    'series_mean_points': 'Mean raw points per contestant in that series (series_total_points / series_size).',
    'pct_of_series_points': "Contestant's share of their series' total points (total_points / series_total_points). The primary apples-to-apples metric; sums to 1.0 within each series.",
    'share_vs_equal': 'pct_of_series_points relative to an even split (x series_size). 1.0 = exactly the average contestant that series; >1 = above the even-split share; <1 = below. Cleanest single cross-series comparison.',
    'points_vs_series_mean': 'total_points / series_mean_points. Same ranking as share_vs_equal, expressed in raw-point space.',
    'series_rank': 'Within-series rank by points, 1 = best (ties take the minimum rank).',
    'is_winner': 'True if the contestant finished first in points that series (series_rank == 1).',
    'pct_gap_to_winner': "Series winner's share minus this contestant's share, in share points (0 for the winner). How far off the series-topping pace, normalized.",
}

notes = '''
Source: Hand-collected total points per contestant per Taskmaster UK series (S1-S21),
reconciled to official full names via the per-series cast list. See SOURCES.md.

KEY CAVEAT: Raw total_points are NOT comparable across series because series differ in
episode count (S1=6, S2-3=5, S4-5=8, S6-21=10) and therefore in points available. Use
pct_of_series_points or share_vs_equal for any cross-series comparison. Points are a
subjective, comedic score awarded on-air, not an objective performance measure.

Scope: UK main series only. Excludes Champion of Champions and New Year Treat specials
and Junior Taskmaster.

License: Scores are facts and not copyrightable; compiled dataset released by the author.
'''

written = package_dataset(df, cfg, name='taskmaster_contestant_metrics_v1',
                          codebook=codebook, notes=notes)
written

---
**Next:** `04-viz.ipynb` — stacked share bar, all-contestant ranking, and supporting charts.

---
## Cleanup

In [ ]:
con.close()
print('connection closed')